In [1]:
import torch
import torch.nn as nn


In [ ]:
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.depthwise = nn.Sequential(
            # Depthwise: mỗi kênh conv riêng (groups=in_channels)
            nn.Conv2d(in_channels, in_channels, kernel_size=3,
                      stride=stride, padding=1, groups=in_channels, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
        )
        self.pointwise = nn.Sequential(
            # Pointwise: kết hợp các kênh lại bằng Conv 1×1
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.pointwise(self.depthwise(x))
# ---- MobileNet V1 ----
class MobileNet(nn.Module):
    def __init__(self, num_classes=1000):
        super().__init__()
        self.model = nn.Sequential(
            # Conv đầu tiên (thường)
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            # 13 khối Depthwise Separable Conv
            DepthwiseSeparableConv(32,  64,  stride=1),
            DepthwiseSeparableConv(64,  128, stride=2),
            DepthwiseSeparableConv(128, 128, stride=1),
            DepthwiseSeparableConv(128, 256, stride=2),
            DepthwiseSeparableConv(256, 256, stride=1),
            DepthwiseSeparableConv(256, 512, stride=2),
            # 5 khối lặp với 512 channels
            DepthwiseSeparableConv(512, 512, stride=1),
            DepthwiseSeparableConv(512, 512, stride=1),
            DepthwiseSeparableConv(512, 512, stride=1),
            DepthwiseSeparableConv(512, 512, stride=1),
            DepthwiseSeparableConv(512, 512, stride=1),
            DepthwiseSeparableConv(512, 1024, stride=2),
            DepthwiseSeparableConv(1024, 1024, stride=1),
            # Global Average Pooling thay cho Flatten lớn
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(1024, num_classes)
    def forward(self, x):
        x = self.model(x)
        x = x.view(x.size(0), -1)   # Flatten: (batch, 1024)
        x = self.classifier(x)
        return x

In [3]:
model = MobileNet(num_classes=1000)
dummy = torch.randn(2, 3, 224, 224)
out = model(dummy)
print(f"Output shape: {out.shape}")    # (2, 1000)
total = sum(p.numel() for p in model.parameters())
print(f"Total params: {total:,}") 

Output shape: torch.Size([2, 1000])
Total params: 4,231,976
